# Colon w/ GO-BS

In [ ]:
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"

import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import math
import time
import random
import inspect
import warnings
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# ----------------------------
# Optional: TabPFN imports
# ----------------------------
try:
    from tabpfn import TabPFNClassifier
except Exception:
    TabPFNClassifier = None

# ----------------------------
# Optional: CatBoost imports
# ----------------------------
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

# ----------------------------
# Optional: torch + TANDEM imports
# ----------------------------
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
except Exception:
    torch = None
    nn = None
    F = None
    DataLoader = None
    TensorDataset = None

# ----------------------------
# BSTabDiff + GO-BS
# ----------------------------
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Global runtime configuration
# ============================================================

GLOBAL_SEED = 42

# After CUDA_VISIBLE_DEVICES="3", physical GPU 3 is cuda:0 inside this process.
REQUESTED_TORCH_DEVICE_STR = "cuda:0"
REQUESTED_CATBOOST_DEVICE = "0"
REQUESTED_TABPFN_DEVICE = "cuda:0"

warnings.filterwarnings("ignore")


# ============================================================
# Fixed best BSTabDiff + GO-BS hyperparameters
# ============================================================

BEST_BSTABDIFF_GOBS_PARAMS = {
    "M": 8,
    "prior_type": "flow",
    "prior_epochs": 300,
    "prior_batch": 64,
    "prior_lr": 0.0032931446587915556,
    "use_ema": True,
    "ema_decay": 0.9925524818123974,

    "n_syn_inner": 64,

    "use_gobs": True,
    "permute_features": False,

    "gobs_num_clusters": 3,
    "gobs_metric": "correlation",
    "gobs_bins": 24,
    "gobs_top_k": 16,
    "gobs_refine_order": False,
    "gobs_direction_select": False,
    "gobs_refine_passes": 3,
    "gobs_boundary_refine_passes": 5,
    "gobs_boundary_window": 16,
    "gobs_lambda_cross": 2.3988509383361634,
    "gobs_gamma_balance": 5.193972052958215e-06,
}

N_SYN = 200


# ============================================================
# Console helper
# ============================================================

def log(msg):
    print(msg, flush=True)


# ============================================================
# Device / seed helpers
# ============================================================

def resolve_torch_device(device_str: str = REQUESTED_TORCH_DEVICE_STR):
    if torch is None:
        return None

    if not torch.cuda.is_available():
        return torch.device("cpu")

    if not device_str.startswith("cuda:"):
        return torch.device(device_str)

    idx = int(device_str.split(":")[1])
    ndev = torch.cuda.device_count()

    if idx >= ndev:
        raise RuntimeError(
            f"Requested {device_str}, but only {ndev} CUDA device(s) are visible."
        )

    return torch.device(device_str)


if torch is not None:
    DEFAULT_TORCH_DEVICE = resolve_torch_device(REQUESTED_TORCH_DEVICE_STR)
    try:
        torch.cuda.set_device(0)
    except Exception:
        pass
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
else:
    DEFAULT_TORCH_DEVICE = None


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)

    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cleanup_torch():
    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass
    gc.collect()


# ============================================================
# TabPFN builder
# ============================================================

def _make_tabpfn(seed=0, tabpfn_device=None, tabpfn_ensembles=16):
    if TabPFNClassifier is None:
        raise ImportError("TabPFN not installed. Try: pip install tabpfn")

    if tabpfn_device is None:
        tabpfn_device = REQUESTED_TABPFN_DEVICE if torch is not None and torch.cuda.is_available() else "cpu"

    sig = inspect.signature(TabPFNClassifier)
    params = sig.parameters
    kwargs = {}

    if "n_estimators" in params:
        kwargs["n_estimators"] = int(tabpfn_ensembles)
    elif "N_ensemble_configurations" in params:
        kwargs["N_ensemble_configurations"] = int(tabpfn_ensembles)

    if "device" in params:
        kwargs["device"] = str(tabpfn_device)

    if "seed" in params:
        kwargs["seed"] = int(seed)
    elif "random_state" in params:
        kwargs["random_state"] = int(seed)

    return TabPFNClassifier(**kwargs)


# ============================================================
# TANDEM
# ============================================================

if torch is not None:

    class GatingNet(nn.Module):
        def __init__(self, input_dim, hidden_dim=128, sigma=0.5):
            super().__init__()
            self.input_dim = input_dim
            self.hidden_dim = hidden_dim
            self.sigma = sigma
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, input_dim),
            )
            self.apply(self.init_weights)
            self._sqrt_2 = math.sqrt(2)

        @staticmethod
        def init_weights(m):
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.001)
                if hasattr(m, "bias") and m.bias is not None:
                    m.bias.data.fill_(0.1)

        def forward(self, x):
            noise = torch.normal(0, self.sigma, size=x.size(), device=x.device) if self.training else 0
            mu = self.net(x)
            z = mu + 0.1 * noise * self.training
            gates = self.hard_sigmoid(z)
            sparse_x = x * gates
            return mu, sparse_x, gates

        @staticmethod
        def hard_sigmoid(x):
            return torch.clamp(x + 0.5, 0.0, 1.0)

        def regularization(self, mu, reduction_func=torch.sum):
            return reduction_func(0.5 + 0.5 * torch.erf(mu / (self.sigma * self._sqrt_2)))

        def get_gates(self, x):
            with torch.no_grad():
                gates = self.hard_sigmoid(self.net(x))
            return gates

        def num_open_gates(self, x):
            return self.get_gates(x).sum(dim=1).cpu().median(dim=0)[0].item()


    class OSDT(nn.Module):
        def __init__(self, input_dim, depth, gating_net=None):
            super().__init__()
            self.input_dim = input_dim
            self.depth = depth
            self.gating_net = gating_net
            self.split_weights = nn.Parameter(torch.randn(depth, input_dim))
            self.feature_thresholds = nn.Parameter(torch.randn(depth))
            self.log_temperatures = nn.Parameter(torch.zeros(depth))

            indices = torch.arange(2**depth)
            bits = ((indices[:, None] & (1 << torch.arange(depth))) > 0).float()
            self.register_buffer("bin_codes", bits.t())

        def forward(self, x):
            all_selected = []
            for i in range(self.depth):
                x_gated = self.gating_net(x)[1] if self.gating_net is not None else x
                split_score = torch.einsum("bi,i->b", x_gated, self.split_weights[i])
                all_selected.append(split_score)

            selected_features = torch.stack(all_selected, dim=1)
            logits = (selected_features - self.feature_thresholds) * torch.exp(-self.log_temperatures)
            logits = torch.stack([-logits, logits], dim=-1)

            bins = torch.sigmoid(logits)
            bins_left = bins[..., 0]
            bins_right = bins[..., 1]

            bin_codes = self.bin_codes.unsqueeze(0)
            bin_match = bins_left.unsqueeze(-1) * (1 - bin_codes) + bins_right.unsqueeze(-1) * bin_codes
            path_probs = bin_match.prod(dim=-2)

            return path_probs


    class OSDTEncoder(nn.Module):
        def __init__(self, input_dim, num_trees, depth, gating_net=None):
            super().__init__()
            self.trees = nn.ModuleList(
                [OSDT(input_dim, depth, gating_net=gating_net) for _ in range(num_trees)]
            )
            self.output_dim = 2**depth

        def forward(self, x):
            outputs = [tree(x) for tree in self.trees]
            outputs = torch.stack(outputs, dim=0)
            return outputs.mean(dim=0)


    class ModularEncoder(nn.Module):
        def __init__(self, input_dim, hidden_layers):
            super().__init__()
            layers = []
            current_in = input_dim

            for h in hidden_layers:
                layers.append(nn.Linear(current_in, h))
                layers.append(nn.LeakyReLU())
                current_in = h

            self.encoder = nn.Sequential(*layers)
            self.output_dim = hidden_layers[-1]

        def forward(self, x):
            return self.encoder(x)


    class ModularDecoder(nn.Module):
        def __init__(self, output_dim, hidden_layers):
            super().__init__()
            layers = []
            current_in = hidden_layers[-1]

            for h in reversed(hidden_layers[:-1]):
                layers.append(nn.Linear(current_in, h))
                layers.append(nn.LeakyReLU())
                current_in = h

            layers.append(nn.Linear(current_in, output_dim))
            self.decoder = nn.Sequential(*layers)

        def forward(self, z):
            return self.decoder(z)


    class Tandem(nn.Module):
        def __init__(self, input_dim, hidden_layers, num_trees=1, osdt_depth=7, gating_net=None):
            super().__init__()
            assert hidden_layers[-1] == 2**osdt_depth, (
                f"The final latent dimension ({hidden_layers[-1]}) must equal 2**osdt_depth ({2**osdt_depth})."
            )
            self.nn_encoder = ModularEncoder(input_dim, hidden_layers)
            self.osdt_encoder = OSDTEncoder(input_dim, num_trees, osdt_depth, gating_net=gating_net)
            self.decoder = ModularDecoder(input_dim, hidden_layers)

        def forward(self, x):
            z_nn = self.nn_encoder(x)
            z_osdt = self.osdt_encoder(x)
            x_rec_nn = self.decoder(z_nn)
            x_rec_osdt = self.decoder(z_osdt)
            return x_rec_nn, x_rec_osdt, z_nn, z_osdt


    class TandemModel(nn.Module):
        def __init__(
            self,
            input_dim,
            n_classes,
            hidden_layers=None,
            num_trees=1,
            osdt_depth=7,
            use_gating=True,
            gating_hidden_dim=128,
            gating_sigma=0.5,
        ):
            super().__init__()

            if hidden_layers is None:
                last_dim = 2**osdt_depth
                hidden_layers = [last_dim * 2, last_dim]

            assert hidden_layers[-1] == 2**osdt_depth

            gating_net = (
                GatingNet(input_dim, hidden_dim=gating_hidden_dim, sigma=gating_sigma)
                if use_gating
                else None
            )

            self.tandem = Tandem(
                input_dim=input_dim,
                hidden_layers=hidden_layers,
                num_trees=num_trees,
                osdt_depth=osdt_depth,
                gating_net=gating_net,
            )
            self.classifier = nn.Linear(hidden_layers[-1], n_classes)

        def forward(self, x):
            x_rec_nn, x_rec_osdt, z_nn, z_osdt = self.tandem(x)
            logits = self.classifier(z_nn)
            return logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt


    class TandemClassifier:
        def __init__(
            self,
            hidden_layers=None,
            num_trees=1,
            osdt_depth=7,
            use_gating=True,
            gating_hidden_dim=128,
            gating_sigma=0.5,
            lr=1e-3,
            weight_decay=1e-5,
            batch_size=32,
            max_epochs=100,
            patience=10,
            rec_weight=1.0,
            align_weight=0.1,
            random_state=42,
            device=None,
        ):
            self.hidden_layers = hidden_layers
            self.num_trees = num_trees
            self.osdt_depth = osdt_depth
            self.use_gating = use_gating
            self.gating_hidden_dim = gating_hidden_dim
            self.gating_sigma = gating_sigma
            self.lr = lr
            self.weight_decay = weight_decay
            self.batch_size = batch_size
            self.max_epochs = max_epochs
            self.patience = patience
            self.rec_weight = rec_weight
            self.align_weight = align_weight
            self.random_state = random_state
            self.device = device if device is not None else (
                DEFAULT_TORCH_DEVICE if DEFAULT_TORCH_DEVICE is not None else torch.device("cpu")
            )
            self.model = None
            self.classes_ = None

        def _build_model(self, n_features, n_classes):
            self.model = TandemModel(
                input_dim=n_features,
                n_classes=n_classes,
                hidden_layers=self.hidden_layers,
                num_trees=self.num_trees,
                osdt_depth=self.osdt_depth,
                use_gating=self.use_gating,
                gating_hidden_dim=self.gating_hidden_dim,
                gating_sigma=self.gating_sigma,
            ).to(self.device)

        def fit(self, X, y, X_val=None, y_val=None):
            set_seed(self.random_state)

            X = np.asarray(X, dtype=np.float32)
            y = np.asarray(y)

            if X_val is None or y_val is None:
                splitter = StratifiedShuffleSplit(
                    n_splits=1,
                    test_size=0.2,
                    random_state=self.random_state,
                )
                (tr_idx, va_idx), = splitter.split(X, y)
                X_tr, X_val = X[tr_idx], X[va_idx]
                y_tr, y_val = y[tr_idx], y[va_idx]
            else:
                X_tr = np.asarray(X, dtype=np.float32)
                X_val = np.asarray(X_val, dtype=np.float32)
                y_tr = np.asarray(y)
                y_val = np.asarray(y_val)

            n_features = X_tr.shape[1]
            classes = np.unique(y_tr)
            self.classes_ = classes
            n_classes = len(classes)

            class_to_idx = {c: i for i, c in enumerate(classes)}
            y_tr_idx = np.vectorize(class_to_idx.get)(y_tr).astype(np.int64)
            y_val_idx = np.vectorize(class_to_idx.get)(y_val).astype(np.int64)

            self._build_model(n_features, n_classes)
            model = self.model

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=self.lr,
                weight_decay=self.weight_decay,
            )
            ce_loss_fn = nn.CrossEntropyLoss()

            X_tr_t = torch.from_numpy(X_tr)
            y_tr_t = torch.from_numpy(y_tr_idx)
            X_val_t = torch.from_numpy(X_val)
            y_val_t = torch.from_numpy(y_val_idx)

            train_ds = TensorDataset(X_tr_t, y_tr_t)
            val_ds = TensorDataset(X_val_t, y_val_t)

            train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True, drop_last=False)
            val_loader = DataLoader(val_ds, batch_size=self.batch_size, shuffle=False, drop_last=False)

            best_state = None
            best_val_loss = float("inf")
            epochs_no_improve = 0

            model.to(self.device)

            for epoch in range(1, self.max_epochs + 1):
                model.train()

                for xb, yb in train_loader:
                    xb = xb.to(self.device)
                    yb = yb.to(self.device)

                    optimizer.zero_grad()
                    logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt = model(xb)

                    ce = ce_loss_fn(logits, yb)
                    rec_nn = F.mse_loss(x_rec_nn, xb)
                    rec_osdt = F.mse_loss(x_rec_osdt, xb)
                    align = F.mse_loss(z_nn, z_osdt)

                    loss = ce + self.rec_weight * (rec_nn + rec_osdt) + self.align_weight * align
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                model.eval()
                val_loss_sum = 0.0
                n_val = 0

                with torch.no_grad():
                    for xb, yb in val_loader:
                        xb = xb.to(self.device)
                        yb = yb.to(self.device)

                        logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt = model(xb)

                        ce = ce_loss_fn(logits, yb)
                        rec_nn = F.mse_loss(x_rec_nn, xb)
                        rec_osdt = F.mse_loss(x_rec_osdt, xb)
                        align = F.mse_loss(z_nn, z_osdt)

                        loss = ce + self.rec_weight * (rec_nn + rec_osdt) + self.align_weight * align

                        bs = xb.size(0)
                        val_loss_sum += loss.item() * bs
                        n_val += bs

                val_loss = val_loss_sum / max(n_val, 1)

                if val_loss < best_val_loss - 1e-6:
                    best_val_loss = val_loss
                    best_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    }
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= self.patience:
                        break

            if best_state is not None:
                model.load_state_dict(best_state)
                model.to(self.device)

            return self

        def _predict_logits(self, X):
            if self.model is None:
                raise RuntimeError("Model not fitted yet.")

            X = np.asarray(X, dtype=np.float32)
            X_t = torch.from_numpy(X)

            ds = TensorDataset(X_t)
            loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False)

            self.model.eval()
            self.model.to(self.device)

            all_logits = []

            with torch.no_grad():
                for (xb,) in loader:
                    xb = xb.to(self.device)
                    logits, _, _, _, _ = self.model(xb)
                    all_logits.append(logits.cpu())

            return torch.cat(all_logits, dim=0).numpy()

        def predict_proba(self, X):
            logits = self._predict_logits(X)
            probs = F.softmax(torch.from_numpy(logits), dim=1).numpy()
            return probs

        def predict(self, X):
            probs = self.predict_proba(X)
            idx = probs.argmax(axis=1)
            return self.classes_[idx] if self.classes_ is not None else idx


# ============================================================
# Classifiers
# ============================================================

def make_clf(
    name="lr",
    seed=0,
    tabpfn_device=None,
    tabpfn_ensembles=16,
    tabpfn_kbest=None,
):
    name = name.lower().strip()

    if name == "lr":
        return Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler(with_mean=True, with_std=True)),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=10000,
                        class_weight="balanced",
                        solver="lbfgs",
                        random_state=seed,
                    ),
                ),
            ]
        )

    if name == "tabpfn":
        if TabPFNClassifier is None:
            raise ImportError("TabPFN not installed. Try: pip install tabpfn")

        from sklearn.feature_selection import SelectKBest, f_classif

        steps = [("imp", SimpleImputer(strategy="median"))]

        if tabpfn_kbest is not None:
            k_val = min(int(tabpfn_kbest), 2000)
            steps.append(("kbest", SelectKBest(score_func=f_classif, k=k_val)))

        steps.append(
            (
                "clf",
                _make_tabpfn(
                    seed=seed,
                    tabpfn_device=tabpfn_device,
                    tabpfn_ensembles=tabpfn_ensembles,
                ),
            )
        )

        return Pipeline(steps)

    if name == "catboost":
        if CatBoostClassifier is None:
            raise ImportError("CatBoost not installed. Try: pip install catboost")

        use_gpu = torch is not None and torch.cuda.is_available()

        params = dict(
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="Logloss",
            iterations=1000,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
        )

        if use_gpu:
            params.update(
                task_type="GPU",
                devices=REQUESTED_CATBOOST_DEVICE,
            )

        cb = CatBoostClassifier(**params)

        return Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("clf", cb),
            ]
        )

    if name == "tandem":
        if torch is None:
            raise ImportError("PyTorch / TANDEM not available.")

        dev = DEFAULT_TORCH_DEVICE if DEFAULT_TORCH_DEVICE is not None else torch.device("cpu")

        clf = TandemClassifier(
            hidden_layers=None,
            num_trees=1,
            osdt_depth=7,
            use_gating=True,
            gating_hidden_dim=128,
            gating_sigma=0.5,
            lr=1e-3,
            weight_decay=1e-5,
            batch_size=32,
            max_epochs=100,
            patience=10,
            rec_weight=1.0,
            align_weight=0.1,
            random_state=seed,
            device=dev,
        )

        return Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler()),
                ("clf", clf),
            ]
        )

    raise ValueError(f"Unknown classifier '{name}'. Choose from: lr, tabpfn, catboost, tandem")


# ============================================================
# Binary metric helpers
# ============================================================

def _predict_proba_binary(clf, X):
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X)
        proba = np.asarray(proba)

        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]

        return proba.ravel()

    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        return 1.0 / (1.0 + np.exp(-s))

    raise ValueError("Classifier has neither predict_proba nor decision_function.")


def _safe_binary_auc_or_acc(clf, X_val, y_val):
    y_val = np.asarray(y_val).astype(int)

    if len(np.unique(y_val)) < 2:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))

    try:
        proba = _predict_proba_binary(clf, X_val)
        return float(roc_auc_score(y_val, proba))
    except Exception:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))


# ============================================================
# Strict 5x5 CV with fold-wise generator training
# ============================================================

def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    clf_name="lr",
    generator_fit_fn=None,
    n_syn=N_SYN,
    do_trtr=True,
    do_c2st=True,
    seed=0,
    tabpfn_device=None,
    tabpfn_ensembles=16,
    tabpfn_kbest=200,
):
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided.")

    X_real = np.asarray(X_real, dtype=np.float32)
    y_real = np.asarray(y_real).astype(int)

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(
        n_splits=5,
        n_repeats=5,
        random_state=seed,
    )

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        log(f"\n[Fold {fold_id:02d}] Fitting BSTabDiff + GO-BS generator...")

        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)

        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)
        X_syn_fold = np.nan_to_num(
            np.asarray(X_syn_fold, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_syn_fold = np.asarray(y_syn_fold).astype(int)

        # Keep only labels valid for current train fold.
        valid_classes = np.unique(ytr)
        mask_valid = np.isin(y_syn_fold, valid_classes)
        X_syn_fold = X_syn_fold[mask_valid]
        y_syn_fold = y_syn_fold[mask_valid]

        # Patch missing classes if synthetic sampling drops one class.
        missing_classes = [c for c in valid_classes if c not in np.unique(y_syn_fold)]
        if len(missing_classes) > 0:
            add_x, add_y = [], []
            for c in missing_classes:
                idx_c = np.where(ytr == c)[0]
                if len(idx_c) > 0:
                    chosen = idx_c[rng.randint(0, len(idx_c))]
                    add_x.append(Xtr[chosen])
                    add_y.append(ytr[chosen])

            if len(add_x) > 0:
                X_syn_fold = np.vstack([X_syn_fold, np.asarray(add_x, dtype=np.float32)])
                y_syn_fold = np.concatenate([y_syn_fold, np.asarray(add_y, dtype=int)])

        if len(np.unique(y_syn_fold)) < 2:
            raise RuntimeError(
                f"Synthetic training data in fold {fold_id} has <2 classes after filtering/fixing."
            )

        # ----------------------------
        # TSTR
        # ----------------------------
        clf_tstr = make_clf(
            clf_name,
            seed=fold_seed,
            tabpfn_device=tabpfn_device,
            tabpfn_ensembles=tabpfn_ensembles,
            tabpfn_kbest=tabpfn_kbest,
        )

        clf_tstr.fit(X_syn_fold, y_syn_fold)

        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)

        tstr_auc.append(roc_auc_score(yte, proba_t))
        tstr_acc.append(accuracy_score(yte, pred_t))

        # ----------------------------
        # TRTR
        # ----------------------------
        if do_trtr:
            clf_trtr = make_clf(
                clf_name,
                seed=fold_seed,
                tabpfn_device=tabpfn_device,
                tabpfn_ensembles=tabpfn_ensembles,
                tabpfn_kbest=tabpfn_kbest,
            )

            clf_trtr.fit(Xtr, ytr)

            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)

            trtr_auc.append(roc_auc_score(yte, proba_r))
            trtr_acc.append(accuracy_score(yte, pred_r))

        # ----------------------------
        # C2ST
        # ----------------------------
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))

            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [
                    np.ones(n_mix, dtype=int),
                    np.zeros(n_mix, dtype=int),
                ]
            )

            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))

            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf("lr", seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)

            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)

            c2st_auc.append(roc_auc_score(ymix_te, proba_m))
            c2st_acc.append(accuracy_score(ymix_te, pred_m))

        log(
            f"[Fold {fold_id:02d}] {clf_name.upper()} | "
            f"TSTR ACC={tstr_acc[-1]:.4f}, AUC={tstr_auc[-1]:.4f}"
        )

        cleanup_torch()

    out = {
        "tstr_auc_mean": float(np.mean(tstr_auc)),
        "tstr_auc_std": float(np.std(tstr_auc)),
        "tstr_acc_mean": float(np.mean(tstr_acc)),
        "tstr_acc_std": float(np.std(tstr_acc)),
        "n_folds": len(tstr_auc),
    }

    if do_trtr and len(trtr_auc) > 0:
        out.update(
            {
                "trtr_auc_mean": float(np.mean(trtr_auc)),
                "trtr_auc_std": float(np.std(trtr_auc)),
                "trtr_acc_mean": float(np.mean(trtr_acc)),
                "trtr_acc_std": float(np.std(trtr_acc)),
            }
        )

    if do_c2st and len(c2st_auc) > 0:
        out.update(
            {
                "c2st_auc_mean": float(np.mean(c2st_auc)),
                "c2st_auc_std": float(np.std(c2st_auc)),
                "c2st_acc_mean": float(np.mean(c2st_acc)),
                "c2st_acc_std": float(np.std(c2st_acc)),
            }
        )

    return out


# ============================================================
# CSV saving helpers
# ============================================================

def save_synth_csv(
    X_syn,
    y_syn=None,
    out_csv="colon_synth_BSTabDiff_n200_gobs_cuda3.csv",
    feature_names=None,
):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]

    assert len(feature_names) == m

    df = pd.DataFrame(X_syn, columns=feature_names)

    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        df.insert(0, "y", y_syn)

    df.to_csv(out_csv, index=False)

    log(f"Saved CSV: {out_csv} | shape={df.shape} | nan_frac={np.isnan(X_syn).mean():.6g}")

    return out_csv


def save_synth_mask_csv(
    R_syn,
    out_csv="colon_synth_BSTabDiff_n200_mask_gobs_cuda3.csv",
):
    R_syn = np.asarray(R_syn).astype(int)

    dfR = pd.DataFrame(
        R_syn,
        columns=[f"r{j}" for j in range(R_syn.shape[1])],
    )

    dfR.to_csv(out_csv, index=False)

    log(f"Saved mask CSV: {out_csv} | shape={dfR.shape} | observed_frac={R_syn.mean():.6g}")

    return out_csv


# ============================================================
# BSTabDiff fixed-parameter fit
# ============================================================

def _fit_bstabdiff_with_fixed_gobs_params(
    Xtr,
    ytr,
    params=BEST_BSTABDIFF_GOBS_PARAMS,
    device=REQUESTED_TORCH_DEVICE_STR,
    seed=0,
    verbose_every=0,
):
    Xtr = np.asarray(Xtr, dtype=np.float32)
    ytr = np.asarray(ytr).astype(int)

    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(Xtr.shape[1])
    ]

    gen = fit_block_subunit_generator(
        X=Xtr,
        feature_specs=feature_specs,
        y=ytr,

        M=params["M"],
        blocks=None,
        permute_features=params["permute_features"],
        prior_type=params["prior_type"],
        device=device,
        seed=seed,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=verbose_every,
        save_dir=None,
        save_name="bstabdiff_colon_gobs_cuda3",
        save_best=True,
        use_ema=params["use_ema"],
        ema_decay=params["ema_decay"],
        return_train_info=False,

        use_gobs=params["use_gobs"],
        gobs_num_clusters=params["gobs_num_clusters"],
        gobs_metric=params["gobs_metric"],
        gobs_bins=params["gobs_bins"],
        gobs_top_k=params["gobs_top_k"],
        gobs_refine_order=params["gobs_refine_order"],
        gobs_direction_select=params["gobs_direction_select"],
        gobs_refine_passes=params["gobs_refine_passes"],
        gobs_boundary_refine_passes=params["gobs_boundary_refine_passes"],
        gobs_boundary_window=params["gobs_boundary_window"],
        gobs_lambda_cross=params["gobs_lambda_cross"],
        gobs_gamma_balance=params["gobs_gamma_balance"],
        gobs_use_cpu_kmeans=False,
    )

    return gen


def make_generator_fit_fn_from_fixed_params(
    params=BEST_BSTABDIFF_GOBS_PARAMS,
    device=REQUESTED_TORCH_DEVICE_STR,
    verbose_every=0,
):
    def _fit(Xtr, ytr, seed=0):
        return _fit_bstabdiff_with_fixed_gobs_params(
            Xtr=np.asarray(Xtr, dtype=np.float32),
            ytr=np.asarray(ytr).astype(int),
            params=params,
            device=device,
            seed=seed,
            verbose_every=verbose_every,
        )

    return _fit


# ============================================================
# Fidelity metric helpers
# ============================================================

def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    nx = len(x)
    ny = len(y)

    if nx == 0 or ny == 0:
        return 0.0, 0.0

    values = np.concatenate([x, y])
    labels = np.concatenate(
        [
            np.zeros(nx, dtype=int),
            np.ones(ny, dtype=int),
        ]
    )

    order = np.argsort(values)

    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1

    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny

    diff = cdf1 - cdf2
    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _rank_transform(X):
    X = np.asarray(X, dtype=float)
    n, m = X.shape

    ranks = np.empty_like(X, dtype=float)

    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=float)

    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=float)

    mean = X.mean(axis=0)
    centered = X - mean

    var = (centered**2).mean(axis=0)

    eps = 1e-8
    std = np.sqrt(var + eps)

    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)

    return mean, var, skew, kurt


def compute_fidelity_metrics(
    X_real,
    y_real,
    X_syn,
    y_syn,
    feature_names=None,
    generator=None,
    random_state=0,
):
    X_real = np.asarray(X_real, dtype=float)
    X_syn = np.asarray(X_syn, dtype=float)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."

    n_features = X_real.shape[1]

    ks_stats = np.zeros(n_features, dtype=float)
    w1_stats = np.zeros(n_features, dtype=float)

    for j in range(n_features):
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    pearson_real = np.corrcoef(X_real, rowvar=False)
    pearson_syn = np.corrcoef(X_syn, rowvar=False)

    mask_offdiag = ~np.eye(n_features, dtype=bool)

    pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]
    pearson_mean = float(np.mean(pearson_abs_diff))
    pearson_max = float(np.max(pearson_abs_diff))

    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)

    spearman_real = np.corrcoef(ranks_real, rowvar=False)
    spearman_syn = np.corrcoef(ranks_syn, rowvar=False)

    spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]
    spearman_mean = float(np.mean(spearman_abs_diff))
    spearman_max = float(np.max(spearman_abs_diff))

    mi_real = mi_syn = mi_abs = None
    mi_mean = mi_max = None

    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)

        try:
            mi_real = mutual_info_classif(
                X_real,
                y_real,
                random_state=random_state,
            )
            mi_syn = mutual_info_classif(
                X_syn,
                y_syn,
                random_state=random_state,
            )
            mi_abs = np.abs(mi_real - mi_syn)
            mi_mean = float(np.mean(mi_abs))
            mi_max = float(np.max(mi_abs))
        except Exception:
            mi_abs = None

    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)

    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    ll_real = ll_syn = None

    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    nn_stats = {}

    try:
        nn_model = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn_model.fit(X_real)

        dists, idx = nn_model.kneighbors(X_syn, return_distance=True)

        dists = dists.ravel()

        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_real": mi_real,
                "mi_syn": mi_syn,
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )

    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# Load dataset
# ============================================================

def load_colon_dataset(
    csv_path=DATA_FILE,
    label_col=TARGET_COL,
):
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. "
            f"Columns: {list(df.columns)[:10]} ..."
        )

    y_raw = df[label_col].values

    X_df = df.drop(columns=[label_col])
    X_df = X_df.select_dtypes(include=[np.number])

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int, copy=False)

    return X, y, list(X_df.columns)


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    log("=" * 80)
    log("BSTabDiff + GO-BS fixed-parameter Colon evaluation")
    log("=" * 80)

    log(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES', '')}")

    if torch is not None and torch.cuda.is_available():
        log(f"Visible CUDA devices: {torch.cuda.device_count()}")
        try:
            log(f"Current CUDA device index: {torch.cuda.current_device()}")
            log(f"Visible device 0 name: {torch.cuda.get_device_name(0)}")
        except Exception:
            pass

    log(f"Using torch device: {DEFAULT_TORCH_DEVICE}")
    log(f"Using CatBoost device: {REQUESTED_CATBOOST_DEVICE}")
    log(f"Using TabPFN device: {REQUESTED_TABPFN_DEVICE}")
    log(f"Fixed use_gobs: {BEST_BSTABDIFF_GOBS_PARAMS['use_gobs']}")
    log(f"Fixed permute_features: {BEST_BSTABDIFF_GOBS_PARAMS['permute_features']}")

    # ----------------------------
    # Load dataset
    # ----------------------------
    X, y, feature_names = load_colon_dataset(
        csv_path=DATA_FILE,
        label_col=TARGET_COL,
    )

    log(
        f"\nLoaded Colon dataset: "
        f"X.shape={X.shape}, y.shape={y.shape}, classes={np.unique(y)}"
    )

    log("\nFixed BSTabDiff + GO-BS parameters:")
    log(str(BEST_BSTABDIFF_GOBS_PARAMS))

    generator_fit_fn = make_generator_fit_fn_from_fixed_params(
        params=BEST_BSTABDIFF_GOBS_PARAMS,
        device=REQUESTED_TORCH_DEVICE_STR,
        verbose_every=0,
    )

    # ============================================================
    # (A) Fit BSTabDiff once on full Colon for reporting
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass

    cpu_before = proc.memory_info().rss if proc is not None else 0

    t0 = time.time()

    gen_full = generator_fit_fn(
        X,
        y,
        seed=GLOBAL_SEED,
    )

    gen_train_time = time.time() - t0

    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)

    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated(device=DEFAULT_TORCH_DEVICE)
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024**3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024**3) if cpu_peak > 0 else 0.0

    log("\n=== BSTabDiff + GO-BS generator: resource usage trained once on full Colon ===")
    log(f"  Train time: {gen_train_time:.2f} sec")
    log(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    log(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    # ----------------------------
    # Sample one synthetic dataset
    # ----------------------------
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)

    X_syn_global = np.nan_to_num(
        np.asarray(X_syn_global, dtype=np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    y_syn_global = np.asarray(y_syn_global).astype(int)

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv="colon_synth_BSTabDiff_n200_gobs_cuda3.csv",
        feature_names=feature_names,
    )

    save_synth_mask_csv(
        R_syn_global,
        out_csv="colon_synth_BSTabDiff_n200_mask_gobs_cuda3.csv",
    )

    # ============================================================
    # (B) 5x5 CV: strict fold-wise BSTabDiff + 4 classifiers
    # ============================================================
    classifiers = [
        ("tabpfn", "TabPFN-2.5"),
        ("tandem", "TANDEM"),
        ("catboost", "CatBoost"),
        ("lr", "Logistic Regression"),
    ]

    summary_rows = []

    for clf_name, clf_label in classifiers:
        log(
            f"\n=== Evaluating STRICT BSTabDiff + GO-BS synthetic + {clf_label} "
            f"(5x5 TSTR/TRTR/C2ST) ==="
        )

        res = eval_5x5_tstr_foldwise_gen(
            X_real=X,
            y_real=y,
            clf_name=clf_name,
            generator_fit_fn=generator_fit_fn,
            n_syn=N_SYN,
            do_trtr=True,
            do_c2st=True,
            seed=0,
            tabpfn_device=REQUESTED_TABPFN_DEVICE if clf_name == "tabpfn" else None,
            tabpfn_ensembles=16,
            tabpfn_kbest=200 if clf_name == "tabpfn" else None,
        )

        log(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
        log(f"  TSTR AUC: {res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}")
        log(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
        log(f"  TRTR AUC: {res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}")
        log(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
        log(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

        summary_rows.append(
            {
                "Classifier": clf_label,
                "TSTR ACC (mean ± std)": f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}",
                "TSTR AUC (mean ± std)": f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}",
                "TRTR ACC (mean ± std)": f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}",
                "TRTR AUC (mean ± std)": f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}",
                "C2ST ACC (mean ± std)": f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}",
                "C2ST AUC (mean ± std)": f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}",
            }
        )

    summary_df = pd.DataFrame(summary_rows)

    summary_path = "bstabdiff_colon_cv_summary_gobs_cuda3.csv"
    summary_df.to_csv(summary_path, index=False)

    log("\n=== Colon: STRICT BSTabDiff + GO-BS synthetic + 4 classifiers: 5x5 summary ===")
    log(summary_df.to_markdown(index=False))
    log(f"Saved CV summary: {summary_path}")

    # ============================================================
    # (C) Fidelity diagnostics
    # ============================================================
    fidelity = compute_fidelity_metrics(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        feature_names=feature_names,
        generator=gen_full,
        random_state=0,
    )

    log("\n=== Colon: Fidelity metrics real vs synthetic, BSTabDiff + GO-BS ===")

    ks_mean = float(np.mean(fidelity["ks_per_feature"]))
    ks_max = float(np.max(fidelity["ks_per_feature"]))

    w1_mean = float(np.mean(fidelity["wasserstein_per_feature"]))
    w1_max = float(np.max(fidelity["wasserstein_per_feature"]))

    log(f"  KS statistic per feature:    mean={ks_mean:.4f}, max={ks_max:.4f}")
    log(f"  Wasserstein-1 per feature:   mean={w1_mean:.4f}, max={w1_max:.4f}")

    log(
        f"  Pearson |corr_real - corr_syn|: "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )

    log(
        f"  Spearman |corr_real - corr_syn|: "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]

    log("  Higher-order moments mean abs diff over features:")
    log(f"    mean:  {higher['mean_abs_diff']:.4e}")
    log(f"    var:   {higher['var_abs_diff']:.4e}")
    log(f"    skew:  {higher['skew_abs_diff']:.4e}")
    log(f"    kurt:  {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        log(
            f"  Mutual information feature-label |ΔMI|: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    if "loglik_real_mean" in fidelity:
        log(
            f"  Log-likelihood if available: "
            f"real={fidelity['loglik_real_mean']}, "
            f"synthetic={fidelity['loglik_syn_mean']}"
        )

    log(
        f"  NaN fraction real: {fidelity['nan_frac_real']:.4e}, "
        f"synthetic: {fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]

        log("  Privacy NN distance from synthetic to nearest real:")
        log(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

    log("\nDone. Fixed BSTabDiff + GO-BS Colon evaluation completed.")

BSTabDiff + GO-BS fixed-parameter Colon evaluation
CUDA_VISIBLE_DEVICES=3
Visible CUDA devices: 1
Current CUDA device index: 0
Visible device 0 name: NVIDIA RTX A6000
Using torch device: cuda:0
Using CatBoost device: 0
Using TabPFN device: cuda:0
Fixed use_gobs: True
Fixed permute_features: False

Loaded Colon dataset: X.shape=(62, 2000), y.shape=(62,), classes=[0 1]

Fixed BSTabDiff + GO-BS parameters:
{'M': 8, 'prior_type': 'flow', 'prior_epochs': 300, 'prior_batch': 64, 'prior_lr': 0.0032931446587915556, 'use_ema': True, 'ema_decay': 0.9925524818123974, 'n_syn_inner': 64, 'use_gobs': True, 'permute_features': False, 'gobs_num_clusters': 3, 'gobs_metric': 'correlation', 'gobs_bins': 24, 'gobs_top_k': 16, 'gobs_refine_order': False, 'gobs_direction_select': False, 'gobs_refine_passes': 3, 'gobs_boundary_refine_passes': 5, 'gobs_boundary_window': 16, 'gobs_lambda_cross': 2.3988509383361634, 'gobs_gamma_balance': 5.193972052958215e-06}

=== BSTabDiff + GO-BS generator: resource usage tr

# GLI w/ GO-BS

In [ ]:
DATA_FILE = "GLI-85_encoded.csv"
TARGET_COL = "label"

import os

# ============================================================
# FORCE THIS PROCESS TO USE ONLY PHYSICAL GPU ID 3
# Must be set BEFORE importing torch / CUDA libraries.
# After masking, physical GPU 3 becomes cuda:0 inside this process.
# ============================================================
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import time
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

try:
    import psutil
except Exception:
    psutil = None

try:
    import torch
except Exception:
    torch = None

from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Global configuration
# ============================================================

GLOBAL_SEED = 42

# After CUDA_VISIBLE_DEVICES="3", physical GPU 3 becomes cuda:0.
REQUESTED_TORCH_DEVICE_STR = "cuda:0"

warnings.filterwarnings("ignore")


# ============================================================
# Fixed best BSTabDiff + GO-BS hyperparameters for GLI
# ============================================================

BEST_BSTABDIFF_GOBS_PARAMS = {
    "M": 8,
    "prior_type": "diffusion",
    "prior_epochs": 800,
    "prior_batch": 32,
    "prior_lr": 0.0003138571708275628,
    "use_ema": True,
    "ema_decay": 0.9910930012968403,

    "n_syn_inner": 64,

    "use_gobs": True,
    "permute_features": False,

    "gobs_num_clusters": 3,
    "gobs_metric": "manhattan",
    "gobs_bins": 32,
    "gobs_top_k": 32,
    "gobs_refine_order": True,
    "gobs_direction_select": False,
    "gobs_refine_passes": 1,
    "gobs_boundary_refine_passes": 0,
    "gobs_boundary_window": 24,
    "gobs_lambda_cross": 2.4372992085995695,
    "gobs_gamma_balance": 9.273654124034148e-06,
}

N_SYN = 200


# ============================================================
# Console helper
# ============================================================

def log(msg):
    print(msg, flush=True)


# ============================================================
# Device / seed helpers
# ============================================================

def resolve_torch_device(device_str=REQUESTED_TORCH_DEVICE_STR):
    if torch is None:
        return None

    if not torch.cuda.is_available():
        return torch.device("cpu")

    if not device_str.startswith("cuda:"):
        return torch.device(device_str)

    idx = int(device_str.split(":")[1])
    ndev = torch.cuda.device_count()

    if idx >= ndev:
        raise RuntimeError(
            f"Requested {device_str}, but only {ndev} CUDA device(s) are visible. "
            f"Because CUDA_VISIBLE_DEVICES='3', physical GPU 3 should appear as cuda:0."
        )

    return torch.device(device_str)


if torch is not None:
    DEFAULT_TORCH_DEVICE = resolve_torch_device(REQUESTED_TORCH_DEVICE_STR)
    try:
        torch.cuda.set_device(0)
    except Exception:
        pass
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
else:
    DEFAULT_TORCH_DEVICE = None


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)

    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cleanup_torch():
    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass
    gc.collect()


# ============================================================
# Classifier: Logistic Regression only
# ============================================================

def make_clf(seed=0):
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    random_state=seed,
                ),
            ),
        ]
    )


# ============================================================
# Metric helpers: binary + multiclass safe
# ============================================================

def _softmax_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(np.sum(ex, axis=1, keepdims=True), 1e-12, None)


def _sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))


def _predict_scores(clf, X):
    if hasattr(clf, "predict_proba"):
        return np.asarray(clf.predict_proba(X))

    if hasattr(clf, "decision_function"):
        scores = np.asarray(clf.decision_function(X))
        if scores.ndim == 1:
            return _sigmoid_numpy(scores)
        return _softmax_numpy(scores)

    raise ValueError("Classifier has neither predict_proba nor decision_function.")


def _predict_labels_from_scores(scores, classes=None):
    scores = np.asarray(scores)

    if scores.ndim == 1:
        pred_idx = (scores >= 0.5).astype(int)
    elif scores.ndim == 2 and scores.shape[1] == 1:
        pred_idx = (scores[:, 0] >= 0.5).astype(int)
    else:
        pred_idx = np.argmax(scores, axis=1)

    if classes is not None:
        classes = np.asarray(classes)
        if len(classes) > 0 and pred_idx.max() < len(classes):
            return classes[pred_idx]

    return pred_idx


def _safe_auc_and_acc(clf, X_val, y_val):
    y_val = np.asarray(y_val).astype(int)
    classes = np.unique(y_val)

    scores = _predict_scores(clf, X_val)

    clf_classes = getattr(clf, "classes_", classes)
    pred = _predict_labels_from_scores(scores, classes=clf_classes)

    acc = float(accuracy_score(y_val, pred))

    if len(classes) < 2:
        return acc, acc

    try:
        if len(classes) == 2:
            if np.ndim(scores) == 2:
                auc = float(roc_auc_score(y_val, scores[:, 1]))
            else:
                auc = float(roc_auc_score(y_val, scores))
        else:
            if np.ndim(scores) != 2 or scores.shape[1] != len(classes):
                return acc, acc

            auc = float(
                roc_auc_score(
                    y_val,
                    scores,
                    multi_class="ovr",
                    average="macro",
                    labels=classes,
                )
            )

        return auc, acc

    except Exception:
        return acc, acc


# ============================================================
# Strict 5x5 CV with fold-wise BSTabDiff training
# Logistic Regression only
# ============================================================

def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    generator_fit_fn,
    n_syn=N_SYN,
    do_trtr=True,
    do_c2st=True,
    seed=0,
):
    X_real = np.asarray(X_real, dtype=np.float32)
    y_real = np.asarray(y_real).astype(int)

    rng = check_random_state(seed)

    rkf = RepeatedStratifiedKFold(
        n_splits=5,
        n_repeats=5,
        random_state=seed,
    )

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        log(f"\n[Fold {fold_id:02d}] Fitting BSTabDiff + GO-BS generator...")
        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)

        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)

        X_syn_fold = np.nan_to_num(
            np.asarray(X_syn_fold, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_syn_fold = np.asarray(y_syn_fold).astype(int)

        # Keep only labels valid for the current train fold.
        valid_classes = np.unique(ytr)
        mask_valid = np.isin(y_syn_fold, valid_classes)
        X_syn_fold = X_syn_fold[mask_valid]
        y_syn_fold = y_syn_fold[mask_valid]

        # If a class is missing after synthetic sampling, minimally patch from real train.
        missing_classes = [c for c in valid_classes if c not in np.unique(y_syn_fold)]

        if len(missing_classes) > 0:
            add_x, add_y = [], []

            for c in missing_classes:
                idx_c = np.where(ytr == c)[0]
                if len(idx_c) > 0:
                    chosen = idx_c[rng.randint(0, len(idx_c))]
                    add_x.append(Xtr[chosen])
                    add_y.append(ytr[chosen])

            if len(add_x) > 0:
                X_syn_fold = np.vstack(
                    [X_syn_fold, np.asarray(add_x, dtype=np.float32)]
                )
                y_syn_fold = np.concatenate(
                    [y_syn_fold, np.asarray(add_y, dtype=int)]
                )

        if len(np.unique(y_syn_fold)) < 2:
            raise RuntimeError(
                f"Synthetic training data in fold {fold_id} has <2 classes after filtering/fixing."
            )

        # TSTR
        clf_tstr = make_clf(seed=fold_seed)
        clf_tstr.fit(X_syn_fold, y_syn_fold)

        auc_t, acc_t = _safe_auc_and_acc(clf_tstr, Xte, yte)
        tstr_auc.append(auc_t)
        tstr_acc.append(acc_t)

        # TRTR
        if do_trtr:
            clf_trtr = make_clf(seed=fold_seed)
            clf_trtr.fit(Xtr, ytr)

            auc_r, acc_r = _safe_auc_and_acc(clf_trtr, Xte, yte)
            trtr_auc.append(auc_r)
            trtr_acc.append(acc_r)

        # C2ST
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))

            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [
                    np.ones(n_mix, dtype=int),
                    np.zeros(n_mix, dtype=int),
                ]
            )

            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))

            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf(seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)

            auc_m, acc_m = _safe_auc_and_acc(clf_c2st, Xmix_te, ymix_te)
            c2st_auc.append(auc_m)
            c2st_acc.append(acc_m)

        log(
            f"[Fold {fold_id:02d}] LR | "
            f"TSTR ACC={tstr_acc[-1]:.4f}, AUC={tstr_auc[-1]:.4f}"
        )

        cleanup_torch()

    out = {
        "tstr_auc_mean": float(np.mean(tstr_auc)),
        "tstr_auc_std": float(np.std(tstr_auc)),
        "tstr_acc_mean": float(np.mean(tstr_acc)),
        "tstr_acc_std": float(np.std(tstr_acc)),
        "n_folds": len(tstr_auc),
    }

    if do_trtr and len(trtr_auc) > 0:
        out.update(
            {
                "trtr_auc_mean": float(np.mean(trtr_auc)),
                "trtr_auc_std": float(np.std(trtr_auc)),
                "trtr_acc_mean": float(np.mean(trtr_acc)),
                "trtr_acc_std": float(np.std(trtr_acc)),
            }
        )

    if do_c2st and len(c2st_auc) > 0:
        out.update(
            {
                "c2st_auc_mean": float(np.mean(c2st_auc)),
                "c2st_auc_std": float(np.std(c2st_auc)),
                "c2st_acc_mean": float(np.mean(c2st_acc)),
                "c2st_acc_std": float(np.std(c2st_acc)),
            }
        )

    return out


# ============================================================
# CSV saving helpers
# ============================================================

def save_synth_csv(
    X_syn,
    y_syn=None,
    out_csv="gli_synth_BSTabDiff_n200_gobs_lr_only_gpu3only.csv",
    feature_names=None,
):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]

    assert len(feature_names) == m

    df = pd.DataFrame(X_syn, columns=feature_names)

    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        df.insert(0, "y", y_syn)

    df.to_csv(out_csv, index=False)

    log(
        f"Saved CSV: {out_csv} | "
        f"shape={df.shape} | "
        f"nan_frac={np.isnan(X_syn).mean():.6g}"
    )

    return out_csv


def save_synth_mask_csv(
    R_syn,
    out_csv="gli_synth_BSTabDiff_n200_mask_gobs_lr_only_gpu3only.csv",
):
    R_syn = np.asarray(R_syn).astype(int)

    dfR = pd.DataFrame(
        R_syn,
        columns=[f"r{j}" for j in range(R_syn.shape[1])],
    )

    dfR.to_csv(out_csv, index=False)

    log(
        f"Saved mask CSV: {out_csv} | "
        f"shape={dfR.shape} | "
        f"observed_frac={R_syn.mean():.6g}"
    )

    return out_csv


# ============================================================
# BSTabDiff fixed GO-BS fit
# ============================================================

def _fit_bstabdiff_with_fixed_gobs_params(
    Xtr,
    ytr,
    params=BEST_BSTABDIFF_GOBS_PARAMS,
    device=REQUESTED_TORCH_DEVICE_STR,
    seed=0,
    verbose_every=0,
):
    Xtr = np.asarray(Xtr, dtype=np.float32)
    ytr = np.asarray(ytr).astype(int)

    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(Xtr.shape[1])
    ]

    gen = fit_block_subunit_generator(
        X=Xtr,
        feature_specs=feature_specs,
        y=ytr,

        M=params["M"],
        blocks=None,
        permute_features=params["permute_features"],
        prior_type=params["prior_type"],
        device=device,
        seed=seed,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=verbose_every,
        save_dir=None,
        save_name="bstabdiff_gli_gobs_fixed_lr_only_gpu3only",
        save_best=True,
        use_ema=params["use_ema"],
        ema_decay=params["ema_decay"],
        return_train_info=False,

        use_gobs=params["use_gobs"],
        gobs_num_clusters=params["gobs_num_clusters"],
        gobs_metric=params["gobs_metric"],
        gobs_bins=params["gobs_bins"],
        gobs_top_k=params["gobs_top_k"],
        gobs_refine_order=params["gobs_refine_order"],
        gobs_direction_select=params["gobs_direction_select"],
        gobs_refine_passes=params["gobs_refine_passes"],
        gobs_boundary_refine_passes=params["gobs_boundary_refine_passes"],
        gobs_boundary_window=params["gobs_boundary_window"],
        gobs_lambda_cross=params["gobs_lambda_cross"],
        gobs_gamma_balance=params["gobs_gamma_balance"],
        gobs_use_cpu_kmeans=False,
    )

    return gen


def make_generator_fit_fn_from_fixed_params(
    params=BEST_BSTABDIFF_GOBS_PARAMS,
    device=REQUESTED_TORCH_DEVICE_STR,
    verbose_every=0,
):
    def _fit(Xtr, ytr, seed=0):
        return _fit_bstabdiff_with_fixed_gobs_params(
            Xtr=np.asarray(Xtr, dtype=np.float32),
            ytr=np.asarray(ytr).astype(int),
            params=params,
            device=device,
            seed=seed,
            verbose_every=verbose_every,
        )

    return _fit


# ============================================================
# Fidelity diagnostics on whole feature set
# ============================================================

def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    nx = len(x)
    ny = len(y)

    if nx == 0 or ny == 0:
        return 0.0, 0.0

    values = np.concatenate([x, y])
    labels = np.concatenate(
        [
            np.zeros(nx, dtype=int),
            np.ones(ny, dtype=int),
        ]
    )

    order = np.argsort(values)

    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1

    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny

    diff = cdf1 - cdf2
    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _rank_transform(X):
    X = np.asarray(X, dtype=np.float32)
    n, m = X.shape

    ranks = np.empty_like(X, dtype=np.float32)

    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=np.float32)

    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=np.float32)

    mean = X.mean(axis=0)
    centered = X - mean

    var = (centered ** 2).mean(axis=0)

    eps = 1e-8
    std = np.sqrt(var + eps)

    skew = (centered ** 3).mean(axis=0) / (std ** 3)
    kurt = (centered ** 4).mean(axis=0) / (std ** 4)

    return mean, var, skew, kurt


def _standardize_for_corr(X):
    X = np.asarray(X, dtype=np.float32)
    Xc = X - X.mean(axis=0, keepdims=True)
    std = Xc.std(axis=0, keepdims=True)
    std_safe = np.where(std < 1e-8, 1.0, std).astype(np.float32)
    Z = Xc / std_safe
    Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return Z


def _corr_abs_diff_blockwise(Z_real, Z_syn, block_size=1024):
    """
    Computes mean/max absolute difference between feature-feature correlations
    without materializing the full m x m correlation matrices.
    """
    Z_real = np.asarray(Z_real, dtype=np.float32)
    Z_syn = np.asarray(Z_syn, dtype=np.float32)

    n, m = Z_real.shape
    denom = max(n - 1, 1)

    total_abs = 0.0
    max_abs = 0.0
    total_count = 0

    for i0 in range(0, m, block_size):
        i1 = min(i0 + block_size, m)

        Zi_r = Z_real[:, i0:i1]
        Zi_s = Z_syn[:, i0:i1]

        for j0 in range(0, m, block_size):
            j1 = min(j0 + block_size, m)

            Zj_r = Z_real[:, j0:j1]
            Zj_s = Z_syn[:, j0:j1]

            Cr = (Zi_r.T @ Zj_r) / denom
            Cs = (Zi_s.T @ Zj_s) / denom

            D = np.abs(Cr - Cs)
            D = np.nan_to_num(D, nan=0.0, posinf=0.0, neginf=0.0)

            if i0 == j0:
                diag_len = i1 - i0
                diag_idx = np.arange(diag_len)
                D[diag_idx, diag_idx] = 0.0
                count = D.size - diag_len
            else:
                count = D.size

            total_abs += float(D.sum())
            max_abs = max(max_abs, float(D.max()))
            total_count += int(count)

            del Cr, Cs, D

        gc.collect()

    mean_abs = total_abs / max(total_count, 1)
    return float(mean_abs), float(max_abs)


def compute_fidelity_metrics_whole(
    X_real,
    y_real,
    X_syn,
    y_syn,
    generator=None,
    random_state=0,
    corr_block_size=1024,
):
    X_real = np.asarray(X_real, dtype=np.float32)
    X_syn = np.asarray(X_syn, dtype=np.float32)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."

    n_features = X_real.shape[1]

    # Marginal distribution distances: all features
    ks_stats = np.zeros(n_features, dtype=np.float32)
    w1_stats = np.zeros(n_features, dtype=np.float32)

    for j in range(n_features):
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    # Full Pearson correlation preservation, blockwise
    log("Computing full Pearson correlation-difference statistics blockwise...")
    Zr = _standardize_for_corr(X_real)
    Zs = _standardize_for_corr(X_syn)

    pearson_mean, pearson_max = _corr_abs_diff_blockwise(
        Zr,
        Zs,
        block_size=corr_block_size,
    )

    del Zr, Zs
    gc.collect()

    # Full Spearman correlation preservation, blockwise
    log("Computing full Spearman correlation-difference statistics blockwise...")
    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)

    Zr_rank = _standardize_for_corr(ranks_real)
    Zs_rank = _standardize_for_corr(ranks_syn)

    spearman_mean, spearman_max = _corr_abs_diff_blockwise(
        Zr_rank,
        Zs_rank,
        block_size=corr_block_size,
    )

    del ranks_real, ranks_syn, Zr_rank, Zs_rank
    gc.collect()

    # Feature-label MI: all features
    mi_abs = None
    mi_mean = None
    mi_max = None

    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)

        try:
            log("Computing mutual information for all features...")
            mi_real = mutual_info_classif(X_real, y_real, random_state=random_state)
            mi_syn = mutual_info_classif(X_syn, y_syn, random_state=random_state)

            mi_abs = np.abs(mi_real - mi_syn)
            mi_mean = float(np.mean(mi_abs))
            mi_max = float(np.max(mi_abs))
        except Exception as e:
            log(f"MI computation failed: {repr(e)}")
            mi_abs = None

    # Higher-order moments: all features
    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)

    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    # Optional likelihood if generator exposes method
    ll_real = ll_syn = None

    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    # Privacy NN distance: whole feature space
    nn_stats = {}

    try:
        nn_model = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn_model.fit(X_real)

        dists, _ = nn_model.kneighbors(
            X_syn,
            return_distance=True,
        )

        dists = dists.ravel()

        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception as e:
        log(f"NN privacy computation failed: {repr(e)}")
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
        "fidelity_feature_count": n_features,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )

    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# Load GLI dataset
# ============================================================

def load_gli_dataset(
    csv_path=DATA_FILE,
    label_col=TARGET_COL,
):
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. "
            f"Columns: {list(df.columns)[:10]} ..."
        )

    y_raw = df[label_col].values

    X_df = df.drop(columns=[label_col])
    X_df = X_df.select_dtypes(include=[np.number])

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int, copy=False)

    return X, y, list(X_df.columns)


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    log("=" * 80)
    log("BSTabDiff + GO-BS GLI fixed-parameter evaluation: LR-only TSTR")
    log("=" * 80)

    log(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES', '')}")

    if torch is not None and torch.cuda.is_available():
        log(f"Visible CUDA devices: {torch.cuda.device_count()}")
        try:
            log(f"Current CUDA device index: {torch.cuda.current_device()}")
            log(f"Visible device 0 name: {torch.cuda.get_device_name(0)}")
        except Exception:
            pass

    log(f"Using torch device: {DEFAULT_TORCH_DEVICE}")

    log("\nFixed BSTabDiff + GO-BS parameters:")
    log(str(BEST_BSTABDIFF_GOBS_PARAMS))

    # ----------------------------
    # Load GLI dataset
    # ----------------------------
    X, y, feature_names = load_gli_dataset(
        csv_path=DATA_FILE,
        label_col=TARGET_COL,
    )

    log(
        f"\nLoaded GLI dataset: "
        f"X.shape={X.shape}, y.shape={y.shape}, classes={np.unique(y)}"
    )

    # ----------------------------
    # Build fixed generator fit function
    # ----------------------------
    generator_fit_fn = make_generator_fit_fn_from_fixed_params(
        params=BEST_BSTABDIFF_GOBS_PARAMS,
        device=REQUESTED_TORCH_DEVICE_STR,
        verbose_every=0,
    )

    # ============================================================
    # (A) Fit BSTabDiff + GO-BS once on full GLI
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass

    cpu_before = proc.memory_info().rss if proc is not None else 0

    t0 = time.time()

    gen_full = generator_fit_fn(
        X,
        y,
        seed=GLOBAL_SEED,
    )

    gen_train_time = time.time() - t0

    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)

    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated(device=DEFAULT_TORCH_DEVICE)
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024 ** 3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024 ** 3) if cpu_peak > 0 else 0.0

    log("\n=== BSTabDiff + GO-BS generator: resource usage trained once on full GLI ===")
    log(f"  Train time: {gen_train_time:.2f} sec")
    log(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    log(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    # ----------------------------
    # Sample one synthetic dataset
    # ----------------------------
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)

    X_syn_global = np.nan_to_num(
        np.asarray(X_syn_global, dtype=np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    y_syn_global = np.asarray(y_syn_global).astype(int)

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv="gli_synth_BSTabDiff_n200_gobs_lr_only_gpu3only.csv",
        feature_names=feature_names,
    )

    save_synth_mask_csv(
        R_syn_global,
        out_csv="gli_synth_BSTabDiff_n200_mask_gobs_lr_only_gpu3only.csv",
    )

    # ============================================================
    # (B) 5x5 CV: strict fold-wise BSTabDiff + LR only
    # ============================================================
    log(
        "\n=== Evaluating STRICT BSTabDiff + GO-BS synthetic + Logistic Regression "
        "(5x5 TSTR/TRTR/C2ST) ==="
    )

    res = eval_5x5_tstr_foldwise_gen(
        X_real=X,
        y_real=y,
        generator_fit_fn=generator_fit_fn,
        n_syn=N_SYN,
        do_trtr=True,
        do_c2st=True,
        seed=0,
    )

    log(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
    log(f"  TSTR AUC: {res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}")
    log(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
    log(f"  TRTR AUC: {res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}")
    log(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
    log(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

    summary_df = pd.DataFrame(
        [
            {
                "Classifier": "Logistic Regression",
                "TSTR ACC (mean ± std)": f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}",
                "TSTR AUC (mean ± std)": f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}",
                "TRTR ACC (mean ± std)": f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}",
                "TRTR AUC (mean ± std)": f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}",
                "C2ST ACC (mean ± std)": f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}",
                "C2ST AUC (mean ± std)": f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}",
            }
        ]
    )

    summary_path = "bstabdiff_gli_cv_summary_gobs_lr_only_gpu3only.csv"
    summary_df.to_csv(summary_path, index=False)

    log("\n=== GLI: STRICT BSTabDiff + GO-BS synthetic + LR-only 5x5 summary ===")
    log(summary_df.to_markdown(index=False))
    log(f"Saved CV summary: {summary_path}")

    # ============================================================
    # (C) Fidelity diagnostics on whole feature set
    # ============================================================
    fidelity = compute_fidelity_metrics_whole(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        generator=gen_full,
        random_state=0,
        corr_block_size=1024,
    )

    log("\n=== GLI: Fidelity metrics real vs synthetic, BSTabDiff + GO-BS ===")
    log(f"  Fidelity feature count: {fidelity['fidelity_feature_count']}")

    ks_mean = float(np.mean(fidelity["ks_per_feature"]))
    ks_max = float(np.max(fidelity["ks_per_feature"]))

    w1_mean = float(np.mean(fidelity["wasserstein_per_feature"]))
    w1_max = float(np.max(fidelity["wasserstein_per_feature"]))

    log(f"  KS statistic per feature:    mean={ks_mean:.4f}, max={ks_max:.4f}")
    log(f"  Wasserstein-1 per feature:   mean={w1_mean:.4f}, max={w1_max:.4f}")

    log(
        f"  Pearson |corr_real - corr_syn| on all features: "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )

    log(
        f"  Spearman |corr_real - corr_syn| on all features: "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]

    log("  Higher-order moments mean abs diff over all features:")
    log(f"    mean:  {higher['mean_abs_diff']:.4e}")
    log(f"    var:   {higher['var_abs_diff']:.4e}")
    log(f"    skew:  {higher['skew_abs_diff']:.4e}")
    log(f"    kurt:  {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        log(
            f"  Mutual information feature-label |ΔMI| on all features: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    if "loglik_real_mean" in fidelity:
        log(
            f"  Log-likelihood if available: "
            f"real={fidelity['loglik_real_mean']}, "
            f"synthetic={fidelity['loglik_syn_mean']}"
        )

    log(
        f"  NaN fraction real: {fidelity['nan_frac_real']:.4e}, "
        f"synthetic: {fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]

        log("  Privacy NN distance from synthetic to nearest real:")
        log(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

    log("\nDone. Fixed BSTabDiff + GO-BS GLI LR-only evaluation completed.")

BSTabDiff + GO-BS GLI fixed-parameter evaluation: LR-only TSTR
CUDA_VISIBLE_DEVICES=3
Visible CUDA devices: 1
Current CUDA device index: 0
Visible device 0 name: NVIDIA RTX A6000
Using torch device: cuda:0

Fixed BSTabDiff + GO-BS parameters:
{'M': 8, 'prior_type': 'diffusion', 'prior_epochs': 800, 'prior_batch': 32, 'prior_lr': 0.0003138571708275628, 'use_ema': True, 'ema_decay': 0.9910930012968403, 'n_syn_inner': 64, 'use_gobs': True, 'permute_features': False, 'gobs_num_clusters': 3, 'gobs_metric': 'manhattan', 'gobs_bins': 32, 'gobs_top_k': 32, 'gobs_refine_order': True, 'gobs_direction_select': False, 'gobs_refine_passes': 1, 'gobs_boundary_refine_passes': 0, 'gobs_boundary_window': 24, 'gobs_lambda_cross': 2.4372992085995695, 'gobs_gamma_balance': 9.273654124034148e-06}

Loaded GLI dataset: X.shape=(85, 22283), y.shape=(85,), classes=[0 1]

=== BSTabDiff + GO-BS generator: resource usage trained once on full GLI ===
  Train time: 54.35 sec
  Approx. peak GPU memory: 7.450 GiB
  A